# French Baby Names

First names registered in France by department, 1900-2020
([dpt2020.csv](https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv)).

1. Evolution over time - line chart with search
2. Regional spread - choropleth map
3. Gender split - centered butterfly chart

In [33]:
# !pip install -r requirements.txt

In [34]:
import json
from urllib.request import urlopen

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Data

Load the CSV, then keep valid years and real names.

In [35]:
url = "https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv"
df = pd.read_csv(url, sep=';', dtype={'annais': str, 'dpt': str})
df.head()

,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


In [36]:
df.columns = ['sex', 'name', 'year', 'dept', 'births']
df = df[df.name != '_PRENOMS_RARES']
df = df[df.year != 'XXXX']
df['year'] = df.year.astype(int)

## 1. Evolution over time

The 300 most common names are shown in grey. Type names separated by `|` to highlight them.

In [37]:
yearly = df.groupby(['name', 'year'], as_index=False).births.sum()

top = yearly.groupby('name').births.sum().nlargest(300).index
pool = yearly[yearly.name.isin(top)]

In [38]:
box = alt.binding(input='text', name='Highlight: ')
search = alt.param(name='q', value='JEAN|MARIE|KEVIN', bind=box)
match = "test(regexp('^(' + q + ')$', 'i'), datum.name)"

base = alt.Chart(pool).encode(
    x='year:Q',
    y=alt.Y('births:Q', scale=alt.Scale(type='log')),
    detail='name:N')

grey = base.mark_line(color='lightgray')
bold = base.mark_line().encode(color='name:N').transform_filter(match)

(grey + bold).add_params(search).properties(width=700, height=400)

alt.LayerChart(...)

## 2. Regional spread

Pick a name and a year. Each department is shaded by how common the name was there (per 1,000 births).

In [39]:
geo_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
with urlopen(geo_url) as r:
    france = json.load(r)

In [40]:
by_dept = df.groupby(['name', 'dept', 'year'], as_index=False).births.sum()
by_dept['total'] = by_dept.groupby(['dept', 'year']).births.transform('sum')
by_dept['per_1000'] = 1000 * by_dept.births / by_dept.total

In [41]:
# the data calls Corsica '20', the map uses 2A and 2B
corse = by_dept[by_dept.dept == '20']
for code in '2A', '2B':
    by_dept = pd.concat([by_dept, corse.assign(dept=code)])

picks = ['KEVIN', 'ERWAN', 'MAEL', 'RONAN', 'AITOR',
         'MAITE', 'MARIE', 'JEAN', 'MOHAMED']
by_dept = by_dept[by_dept.name.isin(picks)]

In [42]:
geo = alt.InlineData(values=france['features'])
fields = ['type', 'geometry', 'properties']
lookup = alt.LookupData(geo, 'properties.code', fields)

menu = alt.binding_select(options=picks, name='Name: ')
slider = alt.binding_range(min=1900, max=2020, step=1, name='Year: ')
name_sel = alt.param(name='sel_name', value='KEVIN', bind=menu)
year_sel = alt.param(name='sel_year', value=1991, bind=slider)

bg = alt.Chart(geo).mark_geoshape(fill='lightgray', stroke='white')

shown = 'datum.name == sel_name && datum.year == sel_year'
front = alt.Chart(by_dept).transform_filter(shown)
front = front.transform_lookup('dept', from_=lookup)
front = front.mark_geoshape(stroke='white').encode(
    color=alt.Color('per_1000:Q', scale=alt.Scale(scheme='blues')),
    tooltip=['properties.nom:N', 'per_1000:Q'])

chart = (bg + front).add_params(name_sel, year_sel)
chart.project('mercator').properties(width=550, height=550)

alt.LayerChart(...)

## 3. Gender split


In [43]:
gender = df.groupby(['name', 'sex', 'year'], as_index=False).births.sum()
gender['decade'] = (gender.year // 20) * 20
gender = gender.groupby(['name', 'sex', 'decade'], as_index=False).births.sum()

bfly_names = [
    'CLAUDE', 'DOMINIQUE', 'CAMILLE', 'ALEXIS', 'MORGAN',
    'ANDREA', 'CHARLIE', 'SACHA', 'ALEX', 'EDEN',
    'MARIE', 'JEAN', 'KEVIN', 'PIERRE', 'NICOLAS',
    'THOMAS', 'LUCAS', 'SOPHIE', 'EMMA', 'LEA'
]
gender = gender[gender.name.isin(bfly_names)]

name_box = alt.binding_select(options=bfly_names, name='Name: ')
name_pick = alt.param(name='bf_name', value='CLAUDE', bind=name_box)

base = (
    alt.Chart(gender)
    .transform_filter('datum.name == bf_name')
    .transform_calculate(
        val='datum.sex == 1 ? -datum.births : datum.births',
        label='datum.sex == 1 ? "Male" : "Female"'
    )
)

bars = base.mark_bar().encode(
    y=alt.Y('decade:O', sort='descending', title=None),
    x=alt.X('val:Q', title='Births', axis=alt.Axis(labelExpr='abs(datum.value)')),
    color=alt.Color('label:N',
        scale=alt.Scale(domain=['Male', 'Female'], range=['steelblue', 'darkorange']),
        legend=alt.Legend(orient='top', title=None)),
    tooltip=[alt.Tooltip('label:N', title='Sex'),
             alt.Tooltip('decade:O', title='Decade'),
             alt.Tooltip('births:Q', title='Births')]
)

bars.add_params(name_pick).properties(width=500, height=300, title='Gender split over time')


alt.Chart(...)